# Experiment 3 · epoch pilot (dorian fold, N=5, seed 0)

Based on `docs/PLAN.md` v16 §8.14: (a) B full fine-tuning, 60 epochs; (b) C training from scratch, 80 epochs; (c) B′ frozen encoder, 60 epochs. After every epoch, single-step RMSE (active cells / land cells) is evaluated on the 10 test runs of the dorian fold, with one curve figure each.
**Estimated 45-55 min** (A100; measured 30 min on 2026-09-18).

**Fonts**: Colab usually has no CJK font; the script automatically tries to install `fonts-noto-cjk` (about 30-60 s the first time); if the log reports that no CJK font was found and labels fall back to English, the figure labels are in English, which is normal; no boxes will appear.
When finished, send the figures from ⑥ and trial_curve.csv to the assistant; the user decides the production epochs. **This notebook does not run the production experiment.**

Order: ① → ② → ③ → ④ → ⑤ (about 45-55 min, runs in the foreground, keep the page open) → ⑥ view figures.

## ① Settings

In [ ]:
FOLD = 'dorian'
DRIVE_ROOT = '/content/drive/MyDrive/Flood2'
PROJ = '/content/Flood_2.0'; DATA_DIR = f'{PROJ}/data/modelB_v1'
EXP2_ROOT = f'{DRIVE_ROOT}/models/exp2'          # Exp. 2 per-fold model_best.pt and norm_stats.json (starting point)
OUT_ROOT = f'{DRIVE_ROOT}/models'                  # Exp. 3 outputs -> OUT_ROOT/exp3/fold_<FOLD>/
FOLD_DIR = f'{OUT_ROOT}/exp3/fold_{FOLD}'
SAMPLES = f'{PROJ}/code/exp3_samples.csv'
EPOCHS = 'B:60,Bp:60,C:80'      # for the pilot; the actual epochs are given by --trial-epochs in ⑤
assert FOLD in ['irma','matthew','ian','milton','dorian','beryl']
print('FOLD =', FOLD, '| output', FOLD_DIR, '| epochs', EPOCHS)

## ② Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for f in ['modelB_v1.tar','modelB_v1_manifest.csv','code/47_exp3_train.py','code/48_exp3_eval_lib.py','code/exp3_samples.csv', f'models/exp2/fold_{FOLD}/model_best.pt', f'models/exp2/fold_{FOLD}/norm_stats.json']:
    p = f'{DRIVE_ROOT}/{f}'; print(('OK  ' if os.path.exists(p) else 'MISSING ') + p)

## ③ Prepare code and data (copy → extract → md5 check; skipped automatically if already done in this session)

In [ ]:
import os, shutil, subprocess, time
os.makedirs(f'{PROJ}/code', exist_ok=True); os.makedirs(f'{PROJ}/data', exist_ok=True)
for f in os.listdir(f'{DRIVE_ROOT}/code'):
    shutil.copy2(f'{DRIVE_ROOT}/code/{f}', f'{PROJ}/code/{f}')
print('Code copied:', sorted(os.listdir(f'{PROJ}/code')))

ok_flag = f'{DATA_DIR}/.verified'
if os.path.exists(ok_flag):
    print('Data already extracted and verified, skipping')
else:
    t0 = time.time()
    shutil.copy2(f'{DRIVE_ROOT}/modelB_v1.tar', '/content/modelB_v1.tar')
    print('tar copied in %.0f s' % (time.time() - t0))
    tar_md5 = subprocess.run(['md5sum', '/content/modelB_v1.tar'], capture_output=True, text=True).stdout.split()[0]
    expect = open(f'{DRIVE_ROOT}/modelB_v1.tar.md5').read().split()[0]
    assert tar_md5 == expect, ('tar md5 mismatch', tar_md5, expect)
    print('tar md5 matches', tar_md5)
    subprocess.run(['tar', '-xf', '/content/modelB_v1.tar', '-C', f'{PROJ}/data'], check=True)
    os.remove('/content/modelB_v1.tar')
    r = subprocess.run(['python3', f'{PROJ}/code/41_pack_modelB_data.py', 'verify', '--data-dir', DATA_DIR,
                        '--manifest', f'{DRIVE_ROOT}/modelB_v1_manifest.csv'], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    assert r.returncode == 0, 'per-file md5 verification failed, stopping'
    open(ok_flag, 'w').write('ok\n')
    print('Extraction + verification done, %.0f s in total' % (time.time() - t0))

## ④ Check GPU (confirm A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, platform
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| python', platform.python_version())
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU: change the runtime type to GPU and rerun from ①')

## ⑤ Run the pilot (foreground, about 45-55 min; the log is also written to Drive)

In [ ]:
import os; os.makedirs(f'{FOLD_DIR}/trial', exist_ok=True)
# Exp. 1 dorian same-event reference lines (result/models/exp1/eval, active cells 0.0149 / land cells 0.0073)
!cd {PROJ} && python3 -u code/47_exp3_train.py --fold {FOLD} --data-dir {DATA_DIR} --exp2-root {EXP2_ROOT} --out-root {OUT_ROOT} --samples {SAMPLES} --trial --trial-epochs B:60,Bp:60,C:80 --ref-exp1-active 0.0149 --ref-exp1-land 0.0073 2>&1 | tee {FOLD_DIR}/trial/trial_stdout.log

## ⑥ View curves (two figures + best epoch per method)

In [ ]:
from IPython.display import Image, display
import csv, os
for k in ('active', 'land'):
    p = f'{FOLD_DIR}/trial/trial_curve_{k}.png'; display(Image(p)) if os.path.exists(p) else print('missing', p)
rows = list(csv.DictReader(open(f'{FOLD_DIR}/trial/trial_curve.csv')))
for m in ('B', 'Bp', 'C'):
    rr = [r for r in rows if r['method'] == m]
    if rr:
        b = min(rr, key=lambda r: float(r['rmse_active']))
        print(f"{m}: {len(rr)} epochs; best active-cell epoch {b['epoch']} RMSE {float(b['rmse_active']):.4f}; last epoch {float(rr[-1]['rmse_active']):.4f}")
print('Send the output above and the two figures to the assistant; do not go on to run the production experiment.')